In [1]:
!pip install langchain langchain_core langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.6/987.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.7/371.7 kB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.5/328.5 kB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.3 MB/s eta 0:00:00


In [2]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass()

··········


In [3]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI

In [4]:
customer_feedback_data = """date,location_id,feedback
2024-01-01,2,"The service was okay, but the coffee was too strong."
2024-01-02,1,"Nice ambiance, but the coffee was a bit cold."
2024-01-02,2,"Loved the new seasonal flavors!"
2024-01-03,1,"The place was dirty and the staff was rude."
2024-01-03,2,"Average experience, nothing special."
2024-01-04,1,"Excellent service and delicious coffee!"
2024-01-04,2,"The coffee was decent, but the pastries were stale."
2024-01-05,1,"Absolutely terrible experience. Never coming back!"
2024-01-05,2,"Great coffee, will definitely return."
2024-01-06,1,"The barista was very knowledgeable and friendly."
2024-01-06,2,"Had a wonderful time with my friends here!"
2024-01-07,2,"It was fine, nothing to complain about."
2024-01-08,1,"Enjoyed the atmosphere and the coffee was perfect."
2024-01-08,2,"The place was a bit crowded, but the coffee was worth it."
2024-01-09,1,"Nice spot to relax and enjoy a good cup of coffee."
2024-01-09,2,"Good experience overall, will come again."
2024-01-10,1,"The staff was not attentive and the coffee was too bitter."
2024-01-10,2,"Loved the decorations and the coffee was great."
2023-11-15,1001,"The coffee was delicious and the service was excellent!"
2023-11-16,1002,"I love the atmosphere here. Great place to relax and work."
2023-11-17,1003,"The barista was friendly and made my latte perfectly."
2023-11-18,1001,"The coffee was cold and the wait was too long."
2023-11-19,1004,"Best coffee shop in town! Highly recommend it."
2023-11-20,1003,"The pastries are amazing, but the coffee was a bit bitter."
2023-11-21,1002,"The robot barista made a mistake with my order."
2023-11-22,1001,"The coffee machine was broken, and there was no coffee available."
2023-11-23,1004,"The music was too loud and made it difficult to work."
2023-11-24,1003,"I found a hair in my coffee."
2023-11-25,1002,"The robot barista was so efficient and friendly!"
2023-11-26,1001,"The coffee was burnt and tasted awful."
2023-11-27,1004,"The bathroom was dirty and unpleasant."
2023-11-28,1002,"I loved the outdoor seating area."
2023-11-29,1003,"The coffee shop was overcrowded and noisy."
2023-11-30,1001,"The barista recommended a great coffee based on my preferences."
2023-12-01,1004,"The robot barista made me laugh with its jokes."
2023-12-02,1002,"The coffee was cold and the service was slow."
2023-12-03,1003,"The coffee was the best I've had in a long time!"
2023-12-04,1001,"The robot barista spilled my coffee on me."
2023-12-05,1004,"Absolute heavenly place! The best tasting coffee I've ever had!"
"""

In [5]:
examples = [
    {"input": "Great coffee and friendly staff!", "output": "Normal"},
    {"input": "The bathroom was dirty and unpleasant", "output": "Unusually negative"},
    {"input": "The coffee was amazing and the service was impeccable.", "output": "Unusually positive"}
]

In [6]:
example_prompt = ChatPromptTemplate.from_messages([("human","{input}"),("ai", "{output}")])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    input_variables=["input"],
    example_prompt=example_prompt,
    examples=examples
    )

In [7]:
print(few_shot_prompt.format())

Human: Great coffee and friendly staff!
AI: Normal
Human: The bathroom was dirty and unpleasant
AI: Unusually negative
Human: The coffee was amazing and the service was impeccable.
AI: Unusually positive


In [8]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data analyst for Wired Brain Coffee, a coffee house chain with over 1600 locations worldwide.
    You will be given feedback submitted from customers. Your job is to classify each feedback as either Normal or Anomalous.
    Anomalous feedback is defined as feedback that is unusually positive or unusually negative."""),
    few_shot_prompt,
    ("human", "{input}")
])

In [9]:
chain = final_prompt | ChatOpenAI(model="gpt-4-turbo", temperature=0)

In [10]:
feedback_rows = customer_feedback_data.strip().split("\n")
for row in feedback_rows[1:]:
    items = row.split(",")
    feedback = items[2]
    cls = chain.invoke({"input": feedback}).content
    print(f"{feedback} -> {cls}")

"The service was okay -> Normal
"Nice ambiance -> Normal
"Loved the new seasonal flavors!" -> Normal
"The place was dirty and the staff was rude." -> Unusually negative
"Average experience -> Normal
"Excellent service and delicious coffee!" -> Normal
"The coffee was decent -> Normal
"Absolutely terrible experience. Never coming back!" -> Unusually negative
"Great coffee -> Normal
"The barista was very knowledgeable and friendly." -> Normal
"Had a wonderful time with my friends here!" -> Normal
"It was fine -> Normal
"Enjoyed the atmosphere and the coffee was perfect." -> Normal
"The place was a bit crowded -> Normal
"Nice spot to relax and enjoy a good cup of coffee." -> Normal
"Good experience overall -> Normal
"The staff was not attentive and the coffee was too bitter." -> Unusually negative
"Loved the decorations and the coffee was great." -> Normal
"The coffee was delicious and the service was excellent!" -> Normal
"I love the atmosphere here. Great place to relax and work." -> Nor